In [3]:
#Tarea 3: Manejo de Valores Nulos y Duplicados

#Objetivo: Implementar estrategias avanzadas para datos faltantes y eliminar duplicados.

### Ejercicios:
#- Analizar patrones de nulos (¿hay correlación entre columnas?).
#- Para `customer_id` nulos, intentar inferir por transacciones de la misma tienda/fecha.
#- Para `amount` nulos, probar diferentes estrategias: media, mediana, valor anterior.
#- Decidir cuál estrategia es mejor y justificar por qué.
#- Eliminar duplicados exactos y por `transaction_id`.
#- Enriquecer datos agregando columnas:
#    - `mes` (extraído de fecha)
#    - `trimestre`
#    - `dia_semana`
#    - `rango_monto` (bajo, medio, alto)
#- Guardar resultado como `transacciones_enriched.csv`.

In [2]:
import os
import sys
from pathlib import Path
from datetime import datetime
# Verificar el intérprete de Python activo
print(f"Intérprete Python: {sys.executable}")
print(f"Versión Python: {sys.version}")

# Mostrar directorio de trabajo actual
current_dir = os.getcwd()
print(f"\nDirectorio actual: {current_dir}")

# Ruta del proyecto dentro del contenedor Docker
# El docker-compose monta ./ en /home/jovyan/work
project_root = Path("/home/jovyan/work")

# Cambiar al directorio del proyecto si es necesario
if os.getcwd() != str(project_root):
    os.chdir(project_root)
    print(f"Directorio cambiado a: {os.getcwd()}")
else:
    print(f"Ya estamos en el directorio del proyecto: {project_root}")

# Agregar el proyecto a sys.path para imports
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print(f"Ruta del proyecto agregada a sys.path")

print(f"\nRaíz del proyecto: {project_root}")


Intérprete Python: /opt/conda/bin/python
Versión Python: 3.11.6 | packaged by conda-forge | (main, Oct  3 2023, 10:40:35) [GCC 12.3.0]

Directorio actual: /home/jovyan/work
Ya estamos en el directorio del proyecto: /home/jovyan/work

Raíz del proyecto: /home/jovyan/work


In [3]:
# Validar que las carpetas existen
# Dentro de Docker: /home/jovyan/work/homeworks/...
homeworks_dir = project_root / "homeworks"
data_dir = homeworks_dir / "data"
output_dir = homeworks_dir / "output"
tarea_3_dir = homeworks_dir / "tarea_3"

print("Validando estructura de carpetas (rutas dentro del contenedor Docker):")
print(f"  project_root : {project_root}")
print(f"  homeworks/   : {homeworks_dir}  → existe: {homeworks_dir.exists()}")
print(f"  data/        : {data_dir}  → existe: {data_dir.exists()}")
print(f"  output/      : {output_dir}  → existe: {output_dir.exists()}")
print(f"  tarea_3/     : {tarea_3_dir}  → existe: {tarea_3_dir.exists()}")

# Crear output si no existe
if not output_dir.exists():
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f"\n✓ Carpeta output creada: {output_dir}")
else:
    print(f"\n✓ Carpeta output ya existe")


Validando estructura de carpetas (rutas dentro del contenedor Docker):
  project_root : /home/jovyan/work
  homeworks/   : /home/jovyan/work/homeworks  → existe: True
  data/        : /home/jovyan/work/homeworks/data  → existe: True
  output/      : /home/jovyan/work/homeworks/output  → existe: True
  tarea_3/     : /home/jovyan/work/homeworks/tarea_3  → existe: True

✓ Carpeta output ya existe


In [4]:
import pandas as pd
import numpy as np
import json
from datetime import datetime
import re

print("✓ Librerías importadas correctamente:")
print(f"  - pandas {pd.__version__}")
print(f"  - numpy {np.__version__}")

✓ Librerías importadas correctamente:
  - pandas 2.1.1
  - numpy 1.24.4


In [5]:
# Cargar datos
csv_path = output_dir / "transacciones_clean.csv"
print(f"Cargando datos desde: {csv_path}")

if csv_path.exists():
    df_raw = pd.read_csv(csv_path)
    print(f"✓ Datos cargados exitosamente")
    print(f"  - Filas: {len(df_raw)}")
    print(f"  - Columnas: {len(df_raw.columns)}")
    print(f"\nPrimeras filas del dataset raw:")
    print(df_raw.head())
else:
    print(f"✗ Archivo no encontrado: {csv_path}")
    print("Verifica que la Tarea 1 fue ejecutada correctamente")

# Guardar copia del original para comparación
df_backup = df_raw.copy()


Cargando datos desde: /home/jovyan/work/homeworks/output/transacciones_clean.csv
✓ Datos cargados exitosamente
  - Filas: 105
  - Columnas: 6

Primeras filas del dataset raw:
  transaction_id        date customer_id  amount       status  \
0      TXN-00001  2023-03-02      1038.0  194.07  DESCONOCIDO   
1      TXN-00002  2023-04-06      1002.0  452.64    CANCELADA   
2      TXN-00003  2023-08-24      1017.0  344.76      FALLIDA   
3      TXN-00004  2023-01-20      1041.0   85.16  DESCONOCIDO   
4      TXN-00005  2023-07-02      1000.0   72.53      FALLIDA   

                    store  
0  TIENDA_SIN_ESPECIFICAR  
1            Tienda_Norte  
2              Tienda_Sur  
3              Tienda_Sur  
4            Tienda_Norte  


In [7]:
#funcion para verificar correlación entre columnas 
#Recorremos los nulos en el df
df_raw_inferedo = df_raw.copy()
for index,row in df_raw[df_raw['customer_id']=="DESCONOCIDO"].iterrows():
    # Intentar varios formatos comunes
    #formatos = ['%Y-%m-%d', '%d/%m/%Y', '%m/%d/%Y', '%Y/%m/%d', '%d-%m-%Y']
    df_raw['date'] = pd.to_datetime(df_raw['date'], errors='coerce')
    fecha_compra=df_raw['date']
    tienda=row['store']
    #Encontrar transacciones en misma tienda, con cliente conocido, en un rango de días (±7)
    mask = (df_raw['store'] == tienda) & (df_raw['customer_id']!="DESCONOCIDO") & (abs((df_raw['date'] - fecha_compra).dt.days) <= 7)
    candidatos = df_raw[mask]['customer_id'].value_counts()
    
    if not candidatos.empty:
        # Asignar el cliente más frecuente en ese contexto
        df_raw_inferedo.loc[index, 'customer_id_inferido'] = candidatos.idxmax()
    else:
        df_raw_inferedo.loc[index, 'customer_id_inferido'] = 'SIN_SUFICIENTES_DATOS'

for index, fila in df_raw_inferedo.iterrows():
    print(f" customer_id :{fila['customer_id']} customer_id_inferido :{fila['customer_id_inferido']}")


 customer_id :1038.0 customer_id_inferido :nan
 customer_id :1002.0 customer_id_inferido :nan
 customer_id :1017.0 customer_id_inferido :nan
 customer_id :1041.0 customer_id_inferido :nan
 customer_id :1000.0 customer_id_inferido :nan
 customer_id :1001.0 customer_id_inferido :nan
 customer_id :1027.0 customer_id_inferido :nan
 customer_id :1034.0 customer_id_inferido :nan
 customer_id :1036.0 customer_id_inferido :nan
 customer_id :1014.0 customer_id_inferido :nan
 customer_id :1039.0 customer_id_inferido :nan
 customer_id :1050.0 customer_id_inferido :nan
 customer_id :DESCONOCIDO customer_id_inferido :1003.0
 customer_id :1047.0 customer_id_inferido :nan
 customer_id :1016.0 customer_id_inferido :nan
 customer_id :1003.0 customer_id_inferido :nan
 customer_id :1007.0 customer_id_inferido :nan
 customer_id :1039.0 customer_id_inferido :nan
 customer_id :1003.0 customer_id_inferido :nan
 customer_id :1004.0 customer_id_inferido :nan
 customer_id :1025.0 customer_id_inferido :nan
 cust

/tmp/ipykernel_2949/1563900878.py:16: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '1003.0' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_raw_inferedo.loc[index, 'customer_id_inferido'] = candidatos.idxmax()


In [8]:
df_raw_inferedo.head()

,transaction_id,date,customer_id,amount,status,store,customer_id_inferido
0,TXN-00001,2023-03-02,1038.0,194.07,DESCONOCIDO,TIENDA_SIN_ESPECIFICAR,NaN
1,TXN-00002,2023-04-06,1002.0,452.64,CANCELADA,Tienda_Norte,NaN
2,TXN-00003,2023-08-24,1017.0,344.76,FALLIDA,Tienda_Sur,NaN
3,TXN-00004,2023-01-20,1041.0,85.16,DESCONOCIDO,Tienda_Sur,NaN
4,TXN-00005,2023-07-02,1000.0,72.53,FALLIDA,Tienda_Norte,NaN


In [9]:
df_raw_inferido=df_raw_inferedo.copy()

In [10]:
df_raw_inferido

,transaction_id,date,customer_id,amount,status,store,customer_id_inferido
0,TXN-00001,2023-03-02,1038.0,194.07,DESCONOCIDO,TIENDA_SIN_ESPECIFICAR,NaN
1,TXN-00002,2023-04-06,1002.0,452.64,CANCELADA,Tienda_Norte,NaN
2,TXN-00003,2023-08-24,1017.0,344.76,FALLIDA,Tienda_Sur,NaN
3,TXN-00004,2023-01-20,1041.0,85.16,DESCONOCIDO,Tienda_Sur,NaN
4,TXN-00005,2023-07-02,1000.0,72.53,FALLIDA,Tienda_Norte,NaN
...,...,...,...,...,...,...,...
100,TXN-00096,2023-08-20,1002.0,225.07,CANCELADA,TIENDA_SIN_ESPECIFICAR,NaN
101,TXN-00097,2023-04-19,1029.0,96.16,DESCONOCIDO,Tienda_Este,NaN
102,TXN-00098,2023-11-02,1018.0,238.43,PENDIENTE,Tienda_Centro,NaN
103,TXN-00099,2023-07-21,DESCONOCIDO,396.91,DESCONOCIDO,Tienda_Este,1018.0
